# Train kinematic-conditioned SD (LoRA) on Google Colab

Use this notebook to run `scripts/train_sd.py` on a **GPU** (CUDA), save checkpoints to **Google Drive**, and copy them back to your Mac / Cursor workspace.

**Before you run**
- **Runtime → Change runtime type → GPU** (T4 / L4 / A100).
- **Dependencies (section 3):** by design this notebook does **not** reinstall PyTorch from the old cu124 index — that often breaks newer Colab runtimes. Install the rest of the stack; only add a `torch`/`torchvision` line if section 2 shows no CUDA.
- **Dataset:** Section 1 sets **`JIGSAWS_ZIP`** to your uploaded zip on Drive; **section 4c** unzips it next to that file and sets **`DATA_ROOT`**. You can instead set **`DATA_ROOT`** to an already-unzipped folder and skip 4c (see the config cell).
- The JIGSAWS root must contain the **suturing** task and **experimental_setup** trees (often **`Suturing/`** and **`Experimental_setup/`** on disk; resolved case-insensitively like `train_sd.py`).
- **Google Drive shortcuts** are unreliable in Colab; a **real zip or folder copy** on Drive is preferred.
- **Code source (pick one):** if **`REPO_URL`** is set, the repo is **cloned** into `CLONE_TARGET` and **`PROJECT_ROOT_DRIVE` is ignored**. If **`REPO_URL`** is empty, the notebook uses **`PROJECT_ROOT_DRIVE`** (a full copy of SS-SD you uploaded to Drive).
- Optional: [Hugging Face token](https://huggingface.co/settings/tokens) if downloads are rate-limited (`Settings → Secrets` in Colab, or set `HF_TOKEN` in a cell).

**After training**
- Checkpoints live under `SAVE_DIR` on Drive (`step_*.pt`, `args.json`). Download that folder in Drive’s web UI, or sync Drive to your machine, and place it under your repo’s `checkpoints/` (e.g. `checkpoints/suturing_expert_lora/`).

## 1. Configuration (edit this cell)

In [ ]:
# --- Paths: edit for your Drive layout ---
DRIVE = "/content/drive/MyDrive"

# JIGSAWS zip on Drive (upload path must match). Section 4c unzips beside this file and sets DATA_ROOT.
JIGSAWS_ZIP = f"{DRIVE}/CS5588: DS Capstone/Surgical Detection/JIGSAWS-20260415T004625Z-3-001.zip"

# Extract to this folder (same directory as the zip; name = zip stem). Created by section 4c.
UNZIP_DIR = f"{DRIVE}/CS5588: DS Capstone/Surgical Detection/JIGSAWS-20260415T004625Z-3-001"

# Filled by section 4c after unzip + auto-detect of JIGSAWS root inside the archive. Leave as-is.
DATA_ROOT = ""

# Alternative: skip the zip — point at the folder that directly contains Suturing/ + Experimental_setup/
# (your layout: .../Surgical Detection/JIGSAWS). Then leave JIGSAWS_ZIP empty and skip section 4c.
# DATA_ROOT = f"{DRIVE}/CS5588: DS Capstone/Surgical Detection/JIGSAWS"
# JIGSAWS_ZIP = ""

# SAVE_DIR: where step_*.pt and args.json are written. Use a *new* folder when you
# change kin_dim or conditioning (enriched run below) so old checkpoints stay separate.
SAVE_DIR = f"{DRIVE}/SS_SD_colab/checkpoints/suturing_enriched_lora"

# DRIVE_OUTPUTS_DIR: mirror the repo's local `outputs/` (grids, eval, diagnostics) — see section 8a.
DRIVE_OUTPUTS_DIR = f"{DRIVE}/SS_SD_colab/outputs_export"

# Optional narrated-eval WAV foley: folder of G1.wav, G2.wav, … (JIGSAWS gesture stems).
# Leave "" to skip unless section 8 finds `PROJECT_ROOT/assets/foley_wavs/`.
FOLEY_DIR = ""  # e.g. f"{DRIVE}/SS_SD_colab/foley_wavs"

# Code: use exactly one workflow —
#   (A) REPO_URL set → git clone into CLONE_TARGET (/content/SS-SD); PROJECT_ROOT_DRIVE is IGNORED.
#   (B) REPO_URL = "" → use PROJECT_ROOT_DRIVE (you must upload the full SS-SD repo there).
# Default: clone from GitHub so you do not need a Drive copy of the code.
REPO_URL = "https://github.com/ango3636/SS-SD.git"
REPO_BRANCH = "fine-tune-sound-effects"  # or "main" for stable baseline
PROJECT_ROOT_DRIVE = f"{DRIVE}/SS-SD"
CLONE_TARGET = "/content/SS-SD"

# Training (defaults mirror typical LoRA runs).
# Professor-feedback enrichments (optional — set tokens to 0 / clip to "" / append False to revert):
#   append_motion_features: +4 scalars (velocity, smoothed vel, accel, jerk) → 80-dim encoder input.
#   num_semantic_tokens: learnable soft prompts for scene semantics (cross-attention sequence grows).
#   clip_scene_prompt: one frozen CLIP text token (after kinematic + semantic tokens).
# num_workers=0: DataLoader workers + Google Drive (FUSE) often hang, spawn errors, or OOM on Colab.
# Use 2 only if DATA_ROOT is on local disk (e.g. copied under /content).
TRAIN_KW = dict(
    expert_only=True,
    train_mode="lora",
    lora_rank=4,
    batch_size=4,
    lr=1e-4,
    epochs=50,
    frame_stride=90,
    image_size=256,
    capture=1,
    save_every=100,
    gradient_accumulation_steps=1,
    num_workers=0,
    model_id="stable-diffusion-v1-5/stable-diffusion-v1-5",
    append_motion_features=True,
    num_semantic_tokens=4,
    clip_scene_prompt="robotic laparoscopic suturing with needle and suture thread",
)

# Optional: Hugging Face token for model download (paste in Colab secrets as HF_TOKEN or uncomment)
import os
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")

## 2. Environment check (GPU)

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("No GPU — enable Runtime → Change runtime type → GPU")

torch: 2.10.0+cu128
cuda available: True
device: Tesla T4


## 3. Install dependencies

In [ ]:
# Do not reinstall torch from the cu124 wheel by default. New Colab GPU runtimes ship torch
# built for CUDA 12.x (often +cu128). Forcing cu124 often breaks CUDA (missing kernels,
# libcudnn errors, or torch.cuda.is_available() == False after deps resolve).
# Run section 2 first: if cuda is True, keep Colab's torch and only install the stack below.
# If cuda is False on a GPU runtime, try (pick one that matches your runtime / section 2 hint):
# %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
# %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu128
# peft needs torchao>=0.16 when an older torchao is present; otherwise get_peft_model() fails.
%pip install -q "torchao>=0.16.0" diffusers transformers accelerate peft safetensors tqdm scikit-learn opencv-python-headless streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 84.2 MB/s eta 0:00:00


## 4. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab — skip Drive mount; set DATA_ROOT / SAVE_DIR to local paths.")

Mounted at /content/drive


## 4b. (Optional) List `MyDrive`

If you are **not** using the zip in section 1, run this after mounting Drive to see folder names and fix paths manually.

In [ ]:
from pathlib import Path

# Run AFTER mounting Drive (section 4).
drive_home = Path("/content/drive/MyDrive")
if not drive_home.is_dir():
    print("Mount Google Drive first (section 4), then re-run this cell.")
else:
    print("Top-level entries under MyDrive (set DATA_ROOT to the JIGSAWS root, not necessarily here):\n")
    for p in sorted(drive_home.iterdir()):
        kind = "dir " if p.is_dir() else "file"
        print(f"  [{kind}] {p.name}")
    print(
        "\nTip: your JIGSAWS root is usually a folder named JIGSAWS or gdrive_cache "
        "containing Suturing, Experimental_setup, etc."
    )

Top-level entries under MyDrive (set DATA_ROOT to the JIGSAWS root, not necessarily here):

  [dir ] CS5567_Deep Learning
  [dir ] CS5588: DS Capstone
  [file] CS5588_Week2_HandsOn_Applied_RAG_NgoAmy.ipynb
  [dir ] Colab Notebooks
  [dir ] My Personal
  [dir ] Old N Important
  [dir ] SS_SD_colab
  [file] SS_SD_colab_ckpt_bundle.zip
  [file] Week4_Ngo.ipynb

Tip: your JIGSAWS root is usually a folder named JIGSAWS or gdrive_cache containing Suturing, Experimental_setup, etc.


## 4c. Unzip JIGSAWS and set `DATA_ROOT`

Run **after** section 4 (and after section 1 so `JIGSAWS_ZIP` / `UNZIP_DIR` are defined). Extracts **`JIGSAWS_ZIP`** into **`UNZIP_DIR`** (skips re-unzip if that folder already contains a valid JIGSAWS root), then sets **`DATA_ROOT`** for sections 6–7.

**Patience:** extracting **to Google Drive** is often **much slower** than to `/content` (20–90+ minutes for large zips). The code cell prints periodic progress while unpacking.

If section 1 sets a non-empty **`DATA_ROOT`** to an existing unzipped folder, this cell only validates it and skips the zip.


In [ ]:
import zipfile
from pathlib import Path


def _find_child_dir_ci(parent: Path, name_lower: str) -> Path | None:
    if not parent.is_dir():
        return None
    try:
        for p in parent.iterdir():
            if p.is_dir() and p.name.lower() == name_lower:
                return p
    except OSError:
        return None
    return None


def _discover_jigsaws_root(base: Path) -> Path:
    if _find_child_dir_ci(base, "suturing") and _find_child_dir_ci(base, "experimental_setup"):
        return base.resolve()
    try:
        children = sorted(p for p in base.iterdir() if p.is_dir())
    except OSError as e:
        raise FileNotFoundError(f"Cannot list {base}: {e}") from e
    for child in children:
        if _find_child_dir_ci(child, "suturing") and _find_child_dir_ci(
            child, "experimental_setup"
        ):
            return child.resolve()
    raise FileNotFoundError(
        f"No folder with Suturing + Experimental_setup under {base.resolve()}. "
        "Is this a full JIGSAWS tree?"
    )


def _extract_zip_progress(zip_path: Path, dest: Path, every: int = 400) -> None:
    """Extract with heartbeat prints — Drive-backed destinations look "stuck" with extractall()."""
    dest.mkdir(parents=True, exist_ok=True)
    nbytes = zip_path.stat().st_size
    print(f"Zip: {zip_path.name} ({nbytes / (1024 ** 2):.1f} MiB) → {dest}")
    print(
        "Google Drive I/O is slow; large archives often take many minutes. "
        f"Printing progress every ~{every} archive entries."
    )
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = zf.infolist()
        n = len(members)
        for i, member in enumerate(members, 1):
            zf.extract(member, dest)
            if i % every == 0 or i == n:
                print(f"  … {i}/{n} entries written")
    print("Unzip finished.")


manual = (DATA_ROOT or "").strip()
if manual:
    DATA_ROOT = str(_discover_jigsaws_root(Path(manual)))
    print("Using manual DATA_ROOT:", DATA_ROOT)
else:
    zip_path = Path((JIGSAWS_ZIP or "").strip())
    dest = Path(UNZIP_DIR)
    root = None
    if dest.is_dir():
        print("Checking existing folder (skip unzip if JIGSAWS root is found)…", dest)
        try:
            root = _discover_jigsaws_root(dest)
        except FileNotFoundError:
            root = None
    if root is None:
        if not zip_path.is_file():
            raise FileNotFoundError(
                f"Zip not found: {zip_path}\n"
                "Upload it to that path, or in section 1 set DATA_ROOT to an existing unzipped JIGSAWS folder."
            )
        _extract_zip_progress(zip_path, dest)
        root = _discover_jigsaws_root(dest)
    DATA_ROOT = str(root)
    print("DATA_ROOT:", DATA_ROOT)


## 5. Resolve project root (`src/`, `scripts/`)

Run **section 4 (Mount Drive)** first if you use Drive paths. With the default **`REPO_URL`**, the repo is **cloned to** `/content/SS-SD` (nothing required under `MyDrive/SS-SD`). Clear **`REPO_URL`** only if you uploaded the full SS-SD tree to **`PROJECT_ROOT_DRIVE`** on Drive.

In [ ]:
import subprocess
from pathlib import Path

_repo = Path(CLONE_TARGET)
if (_repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(_repo), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(_repo), "checkout", REPO_BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(_repo), "pull", "--ff-only", "origin", REPO_BRANCH],
        check=True,
    )
    print("Updated clone:", _repo, "| branch:", REPO_BRANCH)
else:
    print("No repo at", _repo, "yet — the next cell will clone when REPO_URL is set.")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def resolve_project_root() -> Path:
    url = (REPO_URL or "").strip()
    if url:
        target = Path(CLONE_TARGET)
        marker = target / "scripts" / "train_sd.py"
        if marker.is_file():
            return target.resolve()
        if target.exists() and any(target.iterdir()):
            raise FileNotFoundError(
                f"{target} exists but is not a valid SS-SD checkout (missing scripts/train_sd.py). "
                "Delete that folder or change CLONE_TARGET, then re-run."
            )
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, url, str(target)],
            check=True,
        )
        if not marker.is_file():
            raise FileNotFoundError(
                f"Git clone finished but {marker} is missing on GitHub. "
                "Push `scripts/train_sd.py` and `src/suturing_pipeline/data/` from your laptop, then re-run, "
                "or set REPO_URL = '' and upload the full SS-SD tree to PROJECT_ROOT_DRIVE on Drive."
            )
        return target.resolve()

    root = Path(PROJECT_ROOT_DRIVE)
    marker_drive = root / "scripts" / "train_sd.py"
    if not marker_drive.is_file():
        hint = (
            f"No scripts/train_sd.py at {root}.\n"
            "You are using REPO_URL = '' (Drive-only code). Either:\n"
            "  1) Set REPO_URL in section 1 to e.g. https://github.com/ango3636/SS-SD.git "
            "(recommended — clones to /content/SS-SD), or\n"
            "  2) Upload the full SS-SD tree to that Drive path so scripts/train_sd.py exists."
        )
        raise FileNotFoundError(hint)
    return root.resolve()


PROJECT_ROOT = resolve_project_root()

# Older commits imported alignment/loader here; minimal clones lack those files and break --help.
_init = PROJECT_ROOT / "src" / "suturing_pipeline" / "data" / "__init__.py"
if _init.is_file():
    _t = _init.read_text(encoding="utf-8")
    if "from .alignment import" in _t or "from .loader import" in _t:
        _init.write_text(
            '"""Suturing pipeline data package. Import submodules explicitly."""\n',
            encoding="utf-8",
        )
        print("Patched", _init, "(minimal Colab checkout)")

os.chdir(PROJECT_ROOT)

_data_pkg = PROJECT_ROOT / "src" / "suturing_pipeline" / "data" / "jigsaws_dataset.py"
if not _data_pkg.is_file():
    raise FileNotFoundError(
        f"Missing {_data_pkg}. Push the full training stack to GitHub or use a Drive copy of the repo."
    )
_help = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "train_sd.py"), "--help"],
    capture_output=True,
    text=True,
)
if _help.returncode != 0 or "--lora_rank" not in (_help.stdout or ""):
    raise RuntimeError(
        "scripts/train_sd.py --help failed or is outdated.\n"
        f"stdout:\n{_help.stdout}\nstderr:\n{_help.stderr}"
    )
print("PROJECT_ROOT:", PROJECT_ROOT)


## 6. Verify dataset path (`DATA_ROOT`)

Fails fast if `DATA_ROOT` is wrong. Common causes:

- **Shortcut to JIGSAWS** — shortcuts often look empty or missing `suturing/` under Colab. Use a **copied** folder of real files (see intro).
- **One level too shallow** — `DATA_ROOT` must be the folder that **directly** contains the suturing task and experimental-setup trees.
- **Folder name case** — official JIGSAWS zips often use `Suturing/` and `Experimental_setup/`. Linux (Colab) is case-sensitive; this check matches **case-insensitively**, same as `train_sd.py` (`jigsaws_dataset._resolve_ci`).

In [ ]:
from pathlib import Path


def _list_hint(path: Path) -> str:
    if not path.exists():
        return "path does not exist on the mount"
    if not path.is_dir():
        return f"not a directory (symlink/file?): {path}"
    try:
        names = sorted(p.name for p in path.iterdir())
    except OSError as e:
        return f"cannot list directory: {e}"
    head = names[:40]
    extra = f" … (+{len(names) - 40} more)" if len(names) > 40 else ""
    return f"{len(names)} entries: {head}{extra}"


def _find_child_dir_ci(parent: Path, name_lower: str) -> Path | None:
    """Match JIGSAWS layout: e.g. Suturing/ vs suturing/ on case-sensitive Linux."""
    if not parent.is_dir():
        return None
    try:
        for p in parent.iterdir():
            if p.is_dir() and p.name.lower() == name_lower:
                return p
    except OSError:
        return None
    return None


if not (DATA_ROOT or "").strip():
    raise FileNotFoundError(
        "DATA_ROOT is empty. Run section 4c after mounting Drive, or set DATA_ROOT to an unzipped "
        "JIGSAWS folder in section 1."
    )

data = Path(DATA_ROOT)
if not data.is_dir():
    raise FileNotFoundError(
        f"DATA_ROOT is not a directory: {data}\n"
        "Mount Drive (section 4) and run section 4c. If you only added a Drive *shortcut*, copy the real folder into My Drive first.\n"
        f"Debug: {_list_hint(data)}"
    )

required = ("suturing", "experimental_setup")
missing = [key for key in required if _find_child_dir_ci(data, key) is None]
if missing:
    raise FileNotFoundError(
        f"Under DATA_ROOT, expected task folders (any case) for {missing}.\n"
        f"DATA_ROOT={data.resolve()}\n"
        f"Debug listing: {_list_hint(data)}\n\n"
        "If you used a Google Drive *shortcut*: shortcuts often do not mount as real folders in Colab. "
        "Open the dataset in drive.google.com, copy files into a normal My Drive folder, or zip-upload your "
        "local `data/gdrive_cache` and unzip under Drive.\n"
        "If `suturing` appears nested (e.g. DATA_ROOT/JIGSAWS/suturing), set DATA_ROOT one level deeper."
    )

print("DATA_ROOT OK:", data.resolve())
for key in required:
    print(f"  {key!r} ->", _find_child_dir_ci(data, key))

DATA_ROOT OK: /content/drive/MyDrive/CS5588: DS Capstone/Surgical Detection/JIGSAWS-20260415T004625Z-3-001/JIGSAWS
  'suturing' -> /content/drive/MyDrive/CS5588: DS Capstone/Surgical Detection/JIGSAWS-20260415T004625Z-3-001/JIGSAWS/Suturing
  'experimental_setup' -> /content/drive/MyDrive/CS5588: DS Capstone/Surgical Detection/JIGSAWS-20260415T004625Z-3-001/JIGSAWS/Experimental_setup


## 7. Run training

Checkpoints: `SAVE_DIR/step_<N>.pt` and `args.json`. Training uses **CUDA** automatically when available (`train_sd.py`). The training **code cell streams logs live** (progress bars, epoch loss) until the run finishes.

If LoRA setup fails with **`torchao` version** errors, re-run **section 3** so `torchao>=0.16.0` is installed before this section.

**If training fails immediately with CUDA / libcudnn / “no kernel image”:** you likely reinstalled PyTorch with the wrong CUDA wheel. **Runtime → Restart session**, run sections 1–2, keep **section 3’s default** (no torch reinstall), then continue.

**If the run hangs at “Building dataset” or the first epoch:** `DATA_ROOT` on Drive + worker processes is a common cause — the default **`num_workers=0`** in section 1 avoids that. Optionally copy JIGSAWS to `/content` and point `DATA_ROOT` there for faster I/O.


In [ ]:
import torch
print("version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch.version.cuda:", torch.version.cuda)

version: 2.10.0+cu128
cuda available: True
torch.version.cuda: 12.8


In [ ]:
import subprocess
from pathlib import Path

# Match REPO_BRANCH before training (same as section 5 sync).
_repo = Path(CLONE_TARGET)
if (_repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(_repo), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(_repo), "checkout", REPO_BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(_repo), "pull", "--ff-only", "origin", REPO_BRANCH],
        check=True,
    )
    subprocess.run(["git", "-C", str(_repo), "log", "-1", "--oneline"], check=False)
else:
    print("Expected a git clone at", _repo, "— re-run section 5.")


In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

_data_root = Path(str(DATA_ROOT).strip())
if not _data_root.is_dir():
    raise RuntimeError(
        "DATA_ROOT must be a directory (JIGSAWS root with suturing/ + experimental_setup/). "
        "Run section 4c after mounting Drive, or set DATA_ROOT in section 1. "
        f"Invalid: {DATA_ROOT!r}"
    )

os.chdir(PROJECT_ROOT)
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "train_sd.py"),
    "--data_root", str(DATA_ROOT),
    "--save_dir",  str(SAVE_DIR),
    "--device",    "cuda",
]
for k, v in TRAIN_KW.items():
    flag = "--" + k
    if isinstance(v, bool):
        if v:
            cmd.append(flag)
    else:
        cmd.extend([flag, str(v)])

print("Command:", " ".join(shlex.quote(c) for c in cmd))
print("(Streaming logs below — cell runs until training finishes.)\n")
sys.stdout.flush()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["HF_HOME"] = "/content/hf_cache"
env["TRANSFORMERS_CACHE"] = "/content/hf_cache"
env["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Popen + line-by-line read: the only way subprocess output appears in Colab cells.
proc = subprocess.Popen(
    cmd, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
finally:
    proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"train_sd.py exited with code {proc.returncode}.")
print("\nTraining complete.")

## 7b. Post-training evaluation (grid, metrics, video, optional FID)

Run after **section 7** when `step_*.pt` exists. Requires **`PROJECT_ROOT`** (section 5), **`DATA_ROOT`** (section 4c), and **`SAVE_DIR`** (section 1).

Installs **`pytorch-fid`** for optional Fréchet distance. When **`DRIVE_OUTPUTS_DIR`** is set (section 1), grids + video outputs are copied there for archiving.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

EVAL_RUN_NAME = "colab_post_train_eval"
EVAL_CKPT = ""  # or e.g. f"{SAVE_DIR}/step_3300.pt"

%pip install -q pytorch-fid

proj = Path(PROJECT_ROOT)
dr = Path(DATA_ROOT).resolve()
if not dr.is_dir():
    raise RuntimeError("Set DATA_ROOT (section 4c)")
sd = Path(SAVE_DIR)
if not str(EVAL_CKPT).strip():
    ckpts = sorted(sd.glob("step_*.pt"), key=lambda p: p.stat().st_mtime)
    if not ckpts:
        raise RuntimeError(f"No step_*.pt in {sd}. Run section 7 (training) first.")
    ckpt_path = ckpts[-1]
else:
    ckpt_path = Path(EVAL_CKPT)

print("Checkpoint:", ckpt_path)
if not ckpt_path.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

# Pass HF_HOME so SDSampler reuses the model cached by the training cell (no re-download).
env = {
    **os.environ,
    "PYTHONPATH": str(proj / "src"),
    "PYTHONUNBUFFERED": "1",
    "HF_HOME": "/content/hf_cache",
    "TRANSFORMERS_CACHE": "/content/hf_cache",
    "HF_HUB_DISABLE_SYMLINKS_WARNING": "1",
}


def run_script(rel: str, args: list[str]) -> None:
    cmd = [sys.executable, str(proj / "scripts" / rel)] + args
    print("+", " ".join(cmd))
    sys.stdout.flush()
    # Popen + line-by-line read: the only way subprocess output appears in Colab cells.
    proc = subprocess.Popen(
        cmd, cwd=str(proj), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    finally:
        proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{rel} exited {proc.returncode}")


run_script(
    "generate_eval_grid.py",
    [
        "--checkpoint", str(ckpt_path),
        "--data_root", str(dr),
        "--output_dir", str(proj / "outputs" / "eval"),
        "--run_name", EVAL_RUN_NAME,
        "--num_samples", "12",
        "--num_inference_steps", "25",
        "--expert_only",
        "--skip_sweep",
        "--device", "cuda",
    ],
)

eval_dir = proj / "outputs" / "eval" / EVAL_RUN_NAME
run_script(
    "metrics_on_grid.py",
    ["--grid_path", str(eval_dir / "eval_grid.png")],
)

run_script(
    "generate_eval_video.py",
    [
        "--checkpoint", str(ckpt_path),
        "--data_root", str(dr),
        "--output_dir", str(proj / "outputs" / "eval_video"),
        "--run_name", EVAL_RUN_NAME,
        "--device", "cuda",
        "--num_inference_steps", "25",
        "--num_frames", "60",
        "--frame_step", "6",
        "--fps_out", "5",
        "--image_size", "256",
        "--expert_only",
    ],
)

vid_dir = proj / "outputs" / "eval_video" / EVAL_RUN_NAME
run_script(
    "video_quality_metrics.py",
    [
        "--real", str(vid_dir / "real.mp4"),
        "--gen", str(vid_dir / "generated.mp4"),
        "--out_json", str(vid_dir / "video_metrics.json"),
    ],
)

FID_REAL_DIR = ""
FID_GEN_DIR = ""
if FID_REAL_DIR and FID_GEN_DIR:
    rdir, gdir = Path(FID_REAL_DIR), Path(FID_GEN_DIR)
    if rdir.is_dir() and gdir.is_dir():
        run_script(
            "compute_fid.py",
            ["--path1", str(rdir), "--path2", str(gdir), "--device", "cuda"],
        )

if DRIVE_OUTPUTS_DIR:
    out_drive = Path(DRIVE_OUTPUTS_DIR)
    out_drive.mkdir(parents=True, exist_ok=True)
    dest = out_drive / EVAL_RUN_NAME
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(eval_dir, dest / "eval_grid_run", dirs_exist_ok=True)
    shutil.copytree(vid_dir, dest / "eval_video_run", dirs_exist_ok=True)
    print("Copied to", dest)

## 8. Copy checkpoints back to your laptop / Cursor

1. **Google Drive web UI**: open `SAVE_DIR`, select `step_*.pt` and `args.json`, **Download**.
2. In your SS-SD repo locally, put them under e.g. `checkpoints/suturing_expert_lora/` (create folder if needed).
3. Point scripts at the checkpoint: `--checkpoint checkpoints/suturing_expert_lora/step_1000.pt`.

Optional: zip on Colab for one-shot download:

In [ ]:
import shutil
from pathlib import Path

out_zip = Path(DRIVE) / "SS_SD_colab_ckpt_bundle"
shutil.make_archive(str(out_zip), "zip", root_dir=Path(SAVE_DIR).parent, base_dir=Path(SAVE_DIR).name)
print("Wrote:", str(out_zip) + ".zip")

Wrote: /content/drive/MyDrive/SS_SD_colab_ckpt_bundle.zip


## 8a. (Optional) Copy local `outputs/` to Google Drive

Run **after** section 5 (so `PROJECT_ROOT` is set) and **after** you have generated something under the repo’s `outputs/` (e.g. `outputs/eval/…`, `outputs/diagnostics/…`).

The cell copies the entire `PROJECT_ROOT/outputs` tree to **`DRIVE_OUTPUTS_DIR`** (set in section 1), preserving subfolders. It overwrites same-named files on re-run. Mount Drive in section 4 first.

In [ ]:
import shutil
from pathlib import Path

if not Path(DRIVE).is_dir():
    raise RuntimeError("Mount Google Drive first (section 4).")

local_out = Path(PROJECT_ROOT) / "outputs"
drive_out = Path(DRIVE_OUTPUTS_DIR)

if not local_out.is_dir():
    print("Nothing to copy — not a directory:", local_out)
else:
    drive_out.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in local_out.rglob("*"):
        if p.is_file():
            rel = p.relative_to(local_out)
            dest = drive_out / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dest)
            n += 1
    print(f"Copied {n} file(s) to:\n  {drive_out}")

In [ ]:
# --- Eval-video config (narrated) ---
# Checkpoint to render from. By default use the latest step_*.pt in SAVE_DIR.
from pathlib import Path
_ckpts = sorted(Path(SAVE_DIR).glob("step_*.pt"),
                key=lambda p: int(p.stem.split("_")[1]))
EVAL_CKPT = str(_ckpts[-1]) if _ckpts else ""
EVAL_RUN_NAME = "colab_video_v1_narrated"

# Recommendations from the grid-vs-video quality investigation:
#   - 25 inference steps (match eval-grid quality)
#   - flow_warp anchor + init_strength 0.7 (temporal coherence)
#   - start_frame=30 avoids the null-gesture fallback at frame 0
EVAL_VIDEO_KW = dict(
    num_inference_steps=25,
    anchor_mode="flow_warp",       # or "prev_gen" if flow warp drifts
    init_strength=0.7,
    start_frame=30,
    num_frames=60,
    frame_step=6,
    fps_out=5.0,
    image_size=256,
    expert_only=True,               # match training
    enable_narration=True,
    tts_provider="bark",
    tts_voice="v2/en_speaker_6",
    narrate_sidebyside=True,
)

# Pass --foley_dir to generate_eval_video when a folder exists (Drive or repo `assets/foley_wavs`).
_foley_cfg = str(globals().get("FOLEY_DIR", "") or "").strip()
_foley_path = Path(_foley_cfg).expanduser() if _foley_cfg else None
if _foley_path is not None and _foley_path.is_dir():
    EVAL_VIDEO_KW["foley_dir"] = str(_foley_path)
elif _foley_cfg:
    print(
        "WARN: FOLEY_DIR is set but not a directory; skipping WAV foley:",
        _foley_cfg,
    )
else:
    try:
        _repo_foley = Path(PROJECT_ROOT) / "assets" / "foley_wavs"
        if _repo_foley.is_dir():
            EVAL_VIDEO_KW["foley_dir"] = str(_repo_foley)
    except NameError:
        pass

print("EVAL_CKPT:", EVAL_CKPT or "(no checkpoints yet — run section 7)")
if EVAL_VIDEO_KW.get("foley_dir"):
    print("foley_dir:", EVAL_VIDEO_KW["foley_dir"])

EVAL_CKPT: /content/drive/MyDrive/SS_SD_colab/checkpoints/suturing_expert_lora/step_3300.pt


In [ ]:
import os, subprocess, sys
from pathlib import Path

if not EVAL_CKPT:
    raise RuntimeError("No checkpoint selected. Run section 7 first.")

# Bark (suno/bark via HuggingFace) powers --tts_provider bark for narrated exports.
# transformers/scipy/accelerate ship in requirements.txt; moviepy + librosa pull in the rest.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.31.0", "scipy", "accelerate", "moviepy>=2.0.0", "librosa"],
    check=False,
)

os.chdir(PROJECT_ROOT)
out_dir = Path(PROJECT_ROOT) / "outputs" / "eval_video"
out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "generate_eval_video.py"),
    "--checkpoint", EVAL_CKPT,
    "--data_root",  str(DATA_ROOT),
    "--output_dir", str(out_dir),
    "--run_name",   EVAL_RUN_NAME,
    "--device",     "cuda",
]
for k, v in EVAL_VIDEO_KW.items():
    flag = "--" + k
    if isinstance(v, bool):
        if v:
            cmd.append(flag)
    else:
        cmd.extend([flag, str(v)])

print(" ".join(cmd))
_r = subprocess.run(cmd, capture_output=True, text=True)
if _r.stdout: print(_r.stdout, end="")
if _r.stderr: print(_r.stderr, end="", file=sys.stderr)
if _r.returncode != 0:
    raise RuntimeError(f"generate_eval_video.py exited {_r.returncode}")

run_dir = out_dir / EVAL_RUN_NAME
segments_json = run_dir / "narration_segments.json"
if segments_json.exists():
    import json
    data = json.loads(segments_json.read_text())
    print(f"Narration segments: {len(data)}")
    if data:
        print("First narration segment:")
        print(json.dumps(data[0], indent=2)[:1200])

# Copy run artefacts back to Drive so you can download without re-running Colab.
import shutil
drive_out = Path(DRIVE) / "SS_SD_colab" / "eval_video" / EVAL_RUN_NAME
drive_out.mkdir(parents=True, exist_ok=True)
for artifact in run_dir.glob("*"):
    if artifact.is_file() and artifact.suffix.lower() in {".mp4", ".json", ".m4a"}:
        shutil.copy2(artifact, drive_out / artifact.name)
print("Copied to:", drive_out)

/usr/bin/python3 /content/SS-SD/scripts/generate_eval_video.py --checkpoint /content/drive/MyDrive/SS_SD_colab/checkpoints/suturing_expert_lora/step_3300.pt --data_root /content/drive/MyDrive/CS5588: DS Capstone/Surgical Detection/JIGSAWS-20260415T004625Z-3-001/JIGSAWS --output_dir /content/SS-SD/outputs/eval_video --run_name colab_video_v1 --device cuda --num_inference_steps 25 --anchor_mode flow_warp --init_strength 0.7 --start_frame 30 --num_frames 60 --frame_step 6 --fps_out 5.0 --image_size 256 --expert_only
Loading checkpoint: /content/drive/MyDrive/SS_SD_colab/checkpoints/suturing_expert_lora/step_3300.pt
Output dir: /content/SS-SD/outputs/eval_video/colab_video_v1
Building dataset | split=test split_type=onetrialout balance=balanced held_out=None itr=1 capture=1 expert_only=True
  trials in test split: ['Suturing_E004'] (flat_index_size=2926)
  scaler overridden with checkpoint values
Rendering trial: Suturing_E004  (video=Suturing_E004_capture1.avi, frames=2926)
  rendering 60

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


In [ ]:
# Optional: Step-1 narration template debug on one trial segment.
import numpy as np
from pathlib import Path

from suturing_pipeline.audio.narration_templates import (
    build_expert_speed_stats,
    build_narration_payload,
)
from suturing_pipeline.data.data_utils import parse_kinematics, parse_transcription

trial_name = "Suturing_B001"  # change as needed
suffix = trial_name.replace("Suturing_", "")
kin_path = Path(DATA_ROOT) / "suturing" / "kinematics" / "allgestures" / f"Suturing_{suffix}.txt"
tx_path = Path(DATA_ROOT) / "suturing" / "transcriptions" / f"Suturing_{suffix}.txt"

kin = parse_kinematics(kin_path)
segments = parse_transcription(tx_path)
start, end, gesture = segments[0]
segment = kin[start:end + 1]

expert_stats = build_expert_speed_stats(DATA_ROOT)
payload = build_narration_payload(gesture, segment, expert_speed_stats=expert_stats)

print("gesture:", payload["gesture_label"], "-", payload["gesture_description"])
print("speed rating:", payload["summary"]["speed_rating"], "source:", payload["summary"]["speed_rating_source"])
print("narration:")
print(payload["narration_text"])


## 9. Launch Streamlit (GPU inference via Cloudflare tunnel)

Runs `scripts/streamlit_compare.py` **inside this Colab VM** so the generation subprocess picks up the CUDA GPU automatically (the SD sampler auto-resolves `cuda > mps > cpu`).

**Prereqs:** sections 1-5 must have run (so `PROJECT_ROOT`, `DATA_ROOT`, `SAVE_DIR` are defined and the repo is cloned). You also need at least one trained checkpoint under `SAVE_DIR` (section 7) — otherwise the app will show "No trained checkpoints found".

What the cell below does:

1. Symlinks `DATA_ROOT` into `PROJECT_ROOT/data/gdrive_cache` and `SAVE_DIR` into `PROJECT_ROOT/checkpoints/<name>` — those are the fixed paths `streamlit_compare.py` scans.
2. Installs `ffmpeg` (needed for AVI preview transcoding) and `cloudflared`.
3. Starts Streamlit in the background on port 8501.
4. Starts `cloudflared` in the foreground and prints the public `https://...trycloudflare.com` URL — open it in your laptop browser.

Stop the cell (square "interrupt" button) to tear down both processes.

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

os.chdir(PROJECT_ROOT)

# 1. Symlink DATA_ROOT + SAVE_DIR into the fixed locations streamlit_compare.py scans.
def _relink(link: Path, target: Path) -> None:
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink() or link.exists():
        if link.is_symlink() or link.is_file():
            link.unlink()
        else:
            shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)
    print(f"  {link}  ->  {target}")

data_target = Path(DATA_ROOT).resolve()
save_target = Path(SAVE_DIR).resolve()
if not data_target.is_dir():
    raise FileNotFoundError(f"DATA_ROOT not found: {data_target}")
if not save_target.is_dir() or not any(save_target.glob("step_*.pt")):
    raise FileNotFoundError(
        f"SAVE_DIR missing or has no step_*.pt checkpoints: {save_target}. "
        "Run section 7 first."
    )

print("Symlinking data + checkpoints into PROJECT_ROOT:")
_relink(Path(PROJECT_ROOT) / "data" / "gdrive_cache", data_target)
_relink(Path(PROJECT_ROOT) / "checkpoints" / save_target.name, save_target)

# 2. Ensure streamlit is importable in this kernel's Python, then install ffmpeg
#    (apt) and cloudflared (direct binary). Section 3 also pip-installs streamlit,
#    but this cell is self-healing so a fresh runtime that only ran sections 1/4/5
#    can still launch the demo against a previously trained checkpoint.
def _module_available(mod: str) -> bool:
    probe = subprocess.run(
        [sys.executable, "-c", f"import {mod}"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    return probe.returncode == 0

for _mod, _pkg in (("streamlit", "streamlit"), ("cv2", "opencv-python-headless")):
    if not _module_available(_mod):
        print(f"Installing {_pkg} into {sys.executable}...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", _pkg],
            check=True,
        )
if shutil.which("ffmpeg") is None:
    print("Installing ffmpeg...")
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
if shutil.which("cloudflared") is None:
    print("Installing cloudflared...")
    subprocess.run(
        ["wget", "-q",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-O", "/usr/local/bin/cloudflared"],
        check=True,
    )
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
print("ffmpeg:", shutil.which("ffmpeg"))
print("cloudflared:", shutil.which("cloudflared"))

# 3. Kill any prior streamlit/cloudflared from a previous run of this cell.
for name in ("streamlit", "cloudflared"):
    subprocess.run(["pkill", "-f", name], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# 4. Launch Streamlit in the background.
streamlit_log = Path("/content/streamlit.log")
streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "scripts/streamlit_compare.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.address", "0.0.0.0",
        "--browser.gatherUsageStats", "false",
    ],
    cwd=str(PROJECT_ROOT),
    stdout=open(streamlit_log, "w"),
    stderr=subprocess.STDOUT,
)
print(f"streamlit pid={streamlit_proc.pid}, log={streamlit_log}")
time.sleep(5)
if streamlit_proc.poll() is not None:
    print("--- streamlit failed to start; tail of log: ---")
    print(streamlit_log.read_text()[-2000:])
    raise RuntimeError("Streamlit exited immediately. See log above.")

# 5. Foreground cloudflared — prints the public URL, then keeps streaming.
print("\nStarting cloudflared tunnel. The public URL will appear below.\n"
      "Open it in your laptop browser. Interrupt this cell to stop everything.\n")
url_re = re.compile(r"https://[-a-z0-9]+\.trycloudflare\.com")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
public_url = None
try:
    for line in tunnel_proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        if public_url is None:
            m = url_re.search(line)
            if m:
                public_url = m.group(0)
                print(f"\n>>> OPEN THIS IN YOUR BROWSER: {public_url} <<<\n")
except KeyboardInterrupt:
    print("\nStopping cloudflared and streamlit...")
finally:
    for p in (tunnel_proc, streamlit_proc):
        try:
            p.terminate()
            p.wait(timeout=5)
        except Exception:
            p.kill()
    print("Stopped.")

Symlinking data + checkpoints into PROJECT_ROOT:
  /content/SS-SD/data/gdrive_cache  ->  /content/drive/MyDrive/CS5588: DS Capstone/Surgical Detection/JIGSAWS-20260415T004625Z-3-001/JIGSAWS
  /content/SS-SD/checkpoints/suturing_expert_lora  ->  /content/drive/MyDrive/SS_SD_colab/checkpoints/suturing_expert_lora
ffmpeg: /usr/bin/ffmpeg
cloudflared: /usr/local/bin/cloudflared
streamlit pid=19168, log=/content/streamlit.log

Starting cloudflared tunnel. The public URL will appear below.
Open it in your laptop browser. Interrupt this cell to stop everything.

2026-04-18T01:00:17Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you in